# arXiv Parser — Colab (Milestone 2)

Notebook này parse các LaTeX source đã crawl ở Milestone 1 thành ba bảng:
`elements`, `hierarchy`, `bibtex`, kèm `coverage.csv`, `errors.json`,
`checkpoint.json` và `manifest.json`.

## Quy tắc quan trọng

- Parser chỉ **đọc** `runtime/data`, `runtime/progress`, `runtime/exports`.
- Output mặc định là `runtime/parsed/`; có thể đổi bằng `PARSED_OUT`.
- `LIMIT` đếm **publication**, không phải version. `LIMIT=None` chạy toàn bộ.
- Resume chỉ bỏ qua publication đã có checkpoint và đủ ba shard.
- Chạy cell theo thứ tự; không dùng Run all khi chưa kiểm tra `LIMIT`.


## 1. Mount Drive và chọn thư mục dự án

Sửa `PROJECT_DIR` nếu thư mục dự án trên Drive của bạn khác.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = '/content/drive/MyDrive/arxiv'

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    print('Không phải Colab: bỏ qua mount Drive')

root = Path(PROJECT_DIR).expanduser()
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)
print('Project root:', root)


## 2. Lấy/cập nhật code và cài dependency

Cell này fetch `main` và `reset --hard` trong `code/`, nên sửa đổi local trong
`code/` sẽ bị ghi đè. Nếu vừa pull code mới, restart runtime trước khi import lại.


In [ ]:
repo_url = 'https://github.com/KienBuiTan-ichica-1/arxiv-kien.git'
repo = Path('code')
if not (repo / '.git').exists():
    if repo.exists():
        import shutil
        shutil.rmtree(repo)
    subprocess.run(['git', 'clone', repo_url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'remote', 'set-url', 'origin', repo_url], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'src/requirements.txt')], check=True)
sys.path.insert(0, str(repo / 'src'))
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'log', '-1', '--oneline'], text=True).strip())


## 3. Kiểm tra dữ liệu Milestone 1

Chỉ đọc discovery và marker/checksum hiện có, không gọi API và không sửa dữ liệu.


In [ ]:
from crawler.config import load_config
from parser.adapter import list_publications, publications_with_records
from parser import run_parser

config = load_config(Path('code/config/crawler.yaml'))
persistent_root = Path(config.paths['persistent_root']).resolve()
data_root = persistent_root / 'data'
PARSED_OUT = None  # None => persistent_root / 'parsed'

print('Persistent root:', persistent_root)
print('Data root:', data_root)
if not data_root.exists():
    raise FileNotFoundError(f'Chưa có dữ liệu M1 tại {data_root}. Chạy crawl_colab.ipynb trước.')

publications = list_publications(persistent_root)
print('Publications tìm thấy:', len(publications))
for arxiv_id, records in publications_with_records(persistent_root, limit=5):
    print(arxiv_id, '| versions:', [r.version for r in records],
          '| status:', [str(r.status) for r in records])


## 4. Chạy parser

Bắt đầu với `LIMIT=10`. Khi kết quả đúng, đặt `LIMIT=None` để chạy toàn bộ.
`RESUME=True` giữ checkpoint; `RESUME=False` xoá `_parts/` và parse lại từ đầu.

Cell này cần các module A1/A2/A3/B1/B2/B3 đã merge. Nếu chưa, `run_parser`
báo `ParserDependencyError`; test C3 dùng fake services thay thế.


In [ ]:
LIMIT = 10
RESUME = True
if LIMIT is not None and LIMIT <= 0:
    raise ValueError('LIMIT phải > 0 hoặc None')

def log(message):
    print(message, flush=True)

manifest = run_parser(persistent_root, output_root=PARSED_OUT, limit=LIMIT,
                      resume=RESUME, log=log)
out_dir = Path(manifest['output_root'])
print('Đã ghi:', out_dir)


## 5. Thống kê từ manifest

Bao gồm coverage version, số element/BibTeX trước và sau dedup, thời gian chạy,
peak RSS và dung lượng output.


In [ ]:
import json

print(json.dumps({
    'papers_total': manifest['papers_total'],
    'papers_processed': manifest['papers_processed'],
    'versions': manifest['versions'],
    'elements': manifest['elements'],
    'bibtex': manifest['bibtex'],
    'timing': manifest['timing'],
    'memory': manifest['memory'],
    'storage': manifest['storage'],
    'validation': manifest['validation'],
}, indent=2, ensure_ascii=False))
print('Errors:', len(manifest['errors']))
for error in manifest['errors'][:10]:
    print(' ', error)


## 6. Xem ba bảng và coverage

Đọc trực tiếp Parquet bằng pandas; không sửa dữ liệu.


In [ ]:
import pandas as pd

try:
    display
except NameError:
    display = print

elements = pd.read_parquet(out_dir / 'elements.parquet')
hierarchy = pd.read_parquet(out_dir / 'hierarchy.parquet')
bibtex = pd.read_parquet(out_dir / 'bibtex.parquet')
coverage = pd.read_csv(out_dir / 'coverage.csv')

print('elements:', len(elements), '| hierarchy:', len(hierarchy), '| bibtex:', len(bibtex))
display(elements.head(10))
print('Element kinds:')
display(elements['kind'].value_counts())
print('Coverage theo publication:')
display(coverage.head(10))


## 7. Hiệu năng và dung lượng

`peak_rss_bytes` là peak RSS của process (Colab Linux dùng `resource`), không
phải đo riêng từng publication. `parts_bytes` là kích thước shard;
`final_tables_bytes` là output cuối cùng.


In [ ]:
import json
import os

def directory_bytes(path):
    return sum(os.path.getsize(p) for p in Path(path).rglob('*') if p.is_file())

shared = hierarchy.groupby('element_id')['version'].nunique()
print('Elements xuất hiện ở >= 2 version:', int((shared >= 2).sum()))

versions = manifest['versions']
stats = {
    'elapsed_seconds': manifest['timing']['elapsed_seconds'],
    'average_seconds_per_publication': manifest['timing']['average_seconds_per_publication'],
    'peak_rss_bytes': manifest['memory']['peak_rss_bytes'],
    'final_tables_bytes': directory_bytes(out_dir),
    'parts_bytes': manifest['storage']['parts_bytes'],
    'parse_success_rate_versions': (
        versions['versions_parsed'] / versions['versions_total']
        if versions['versions_total'] else None),
}
print(json.dumps(stats, indent=2, ensure_ascii=False))


## Ghi chú vận hành

### Output
- Bảng cuối: `runtime/parsed/{elements,hierarchy,bibtex}.parquet`
- Thống kê: `coverage.csv`, `errors.json`, `manifest.json`, `checkpoint.json`
- Validation: `manifest['validation']` (schema, unique id, hierarchy FK, citation refs)
- Shard tạm: `runtime/parsed/_parts/`
- Không ghi vào `runtime/data`, `runtime/progress`, `runtime/exports`.

### Resume
- Publication chỉ bị bỏ qua khi có trong checkpoint và đủ ba shard.
- `RESUME=False` xoá checkpoint + `_parts/` rồi parse lại; dùng khi đổi logic parser.
- Version `no_tex` được ghi nhận là empty, không bị bỏ khỏi coverage.
- Version `failed`/`missing` được liệt kê trong `errors.json`, không dừng run.

### Khi nào cần chạy lại
- Sau khi crawler thêm version mới hoặc sửa source: chạy `RESUME=True` để parse phần mới.
- Sau khi đổi code parser/cleanup/dedup: chạy `RESUME=False` để tránh trộn output cũ.
- CLI tương đương: `PYTHONPATH=src python -m parser.cli --persistent-root runtime --limit 10`.
